# Step 19 — Specific and Broader harmonized cell annotations over H&E

This notebook overlays the two completed Step 18 annotations on the **same
cropped 0.3-µm/pixel H&E image used for StarDist**:

```python
Specific_celltype_annotation
Broader_celltype_annotation
```

The StarDist H&E crop and its fitted mask-to-Proseg affine transform are reused,
so no new image registration is performed.

## Why this notebook is both automatic and customizable

The notebook automatically generates whole-capture-area views for all 12
samples. Close-up views are controlled through one editable `ROI_SPECS`
dictionary, so regions can be adjusted without rerunning the whole-slide
figures.

For every sample it can create:

```text
Specific annotation over H&E
Broader annotation over H&E
Specific-versus-Broader side-by-side view
```

For every named ROI it can create the same three outputs.

## Default display settings

```python
CELL_ALPHA = 1
WHOLE_POINT_SIZE = 3.0
ROI_POINT_SIZE = 16.0
```

The 50% point transparency keeps the H&E visible. Point sizes, image brightness,
contrast, legends, and output resolution are all configurable.

## ROI definitions

ROIs can be specified by:

```python
# Exact image-pixel bounds
{"bounds": (left, upper, right, lower)}

# Center at a fractional image position
{
    "center_fraction": (0.67, 0.25),
    "size": 2500,
}

# Center at an exact image-pixel position
{
    "center_px": (7000, 3000),
    "width": 2200,
    "height": 1800,
}
```

The notebook writes coordinate-grid overviews to help choose useful regions.

## Coordinate convention

All ROI coordinates refer to the cropped StarDist H&E image:

```text
x = 0 at the left edge
y = 0 at the top edge
```

The “whole-slide” outputs in this notebook are the whole StarDist/Visium
capture-area H&E crop, not necessarily the entire original scanner slide.


In [1]:
# ---------------------------------------------------------------------
# Imports and configuration
# ---------------------------------------------------------------------
from __future__ import annotations

import gc
import json
import math
import re
import warnings
from collections import OrderedDict
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from PIL import Image, ImageEnhance
from IPython.display import display

Image.MAX_IMAGE_PIXELS = None
plt.ioff()

PROJECT_ROOT = Path(
    "/host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057"
)
TMP_ROOT = PROJECT_ROOT / "tmp"
PIPELINE_ROOT = (
    TMP_ROOT
    / "proseg_resolvi_immune_enrichment_v1"
)

STEP18_ROOT = (
    PIPELINE_ROOT
    / "18_harmonized_signature_and_tumor_cluster_annotations"
)
ANNOTATION_HANDOFF_PATH = (
    STEP18_ROOT
    / "all_cells_mixed_annotation_handoff.parquet"
)
PALETTE_JSON_PATH = (
    STEP18_ROOT
    / "celltype_color_palette.json"
)
PALETTE_CSV_PATH = (
    STEP18_ROOT
    / "celltype_color_palette.csv"
)

MERGED_HANDOFF_FILENAME = (
    "VisiumHD_12samples_ResolVI_corrected_preliminary_annotation.zarr"
)

OUTPUT_ROOT = (
    PIPELINE_ROOT
    / "19_harmonized_annotations_over_HE"
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SPECIFIC_COLUMN = "Specific_celltype_annotation"
BROADER_COLUMN = "Broader_celltype_annotation"

# ------------------------------------------------------------------
# Point and H&E appearance
# ------------------------------------------------------------------
CELL_ALPHA = 1.0

# User requested a relatively visible point size. Whole-slide plots can be
# reduced to 2–3 if a particularly dense specimen becomes visually crowded.
WHOLE_POINT_SIZE = 5.0
ROI_POINT_SIZE = 20.0

HE_BRIGHTNESS = 1.00
HE_CONTRAST = 1.00
HE_COLOR_SATURATION = 1.00

WHOLE_PREVIEW_MAX_SIDE = 6000
WHOLE_FIGURE_MAX_INCHES = 14.0
ROI_FIGURE_MAX_INCHES = 10.0

PLOT_DPI = 350
SHOW_LEGEND = True
LEGEND_FONT_SIZE = 8
LEGEND_COLUMNS_WHOLE = 2
LEGEND_COLUMNS_COMPARISON = 5

# Draw abundant populations first so sparse populations remain visible.
DRAW_ABUNDANT_FIRST = True

# Coordinate validation.
MIN_CELLS_IN_IMAGE_FRACTION = 0.95

# ------------------------------------------------------------------
# Output switches
# ------------------------------------------------------------------
RUN_WHOLE_SLIDE_PLOTS = True
RUN_ROI_SELECTION_OVERVIEWS = True
RUN_CLOSEUP_PLOTS = True

MAKE_SEPARATE_SPECIFIC = True
MAKE_SEPARATE_BROADER = True
MAKE_SIDE_BY_SIDE = True

OVERWRITE = False

PIPELINE_VERSION = (
    "2026-08-21-harmonized-annotations-over-HE-v1"
)

print("Annotation handoff:", ANNOTATION_HANDOFF_PATH)
print("Output root:", OUTPUT_ROOT)
print("Cell alpha:", CELL_ALPHA)
print("Whole point size:", WHOLE_POINT_SIZE)
print("ROI point size:", ROI_POINT_SIZE)


Annotation handoff: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/18_harmonized_signature_and_tumor_cluster_annotations/all_cells_mixed_annotation_handoff.parquet
Output root: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/19_harmonized_annotations_over_HE
Cell alpha: 1.0
Whole point size: 5.0
ROI point size: 20.0


In [2]:
# ---------------------------------------------------------------------
# Sample order and image/affine locations
# ---------------------------------------------------------------------
SAMPLE_INFO = OrderedDict(
    {
        # Melanoma
        "Screen_16_22": {
            "patient": "patient_16_22",
            "cancer_type": "melanoma",
            "biopsy_stage": "Screen",
        },
        "C2D15_16_22": {
            "patient": "patient_16_22",
            "cancer_type": "melanoma",
            "biopsy_stage": "C2D15",
        },
        "Screen_18_23": {
            "patient": "patient_18_23",
            "cancer_type": "melanoma",
            "biopsy_stage": "Screen",
        },
        "C2D15_18_23": {
            "patient": "patient_18_23",
            "cancer_type": "melanoma",
            "biopsy_stage": "C2D15",
        },
        "Screen_30_16": {
            "patient": "patient_30_16",
            "cancer_type": "melanoma",
            "biopsy_stage": "Screen",
        },
        "C2D15_30_16": {
            "patient": "patient_30_16",
            "cancer_type": "melanoma",
            "biopsy_stage": "C2D15",
        },

        # NSCLC
        "Screen_17_26": {
            "patient": "patient_17_26",
            "cancer_type": "NSCLC",
            "biopsy_stage": "Screen",
        },
        "C2D15_17_26": {
            "patient": "patient_17_26",
            "cancer_type": "NSCLC",
            "biopsy_stage": "C2D15",
        },
        "Screen_39_21": {
            "patient": "patient_39_21",
            "cancer_type": "NSCLC",
            "biopsy_stage": "Screen",
        },
        "C2D15_39_21": {
            "patient": "patient_39_21",
            "cancer_type": "NSCLC",
            "biopsy_stage": "C2D15",
        },

        # MSS-CRC
        "Screen_23_25": {
            "patient": "patient_23_25",
            "cancer_type": "colon_cancer",
            "biopsy_stage": "Screen",
        },
        "C2D15_23_25": {
            "patient": "patient_23_25",
            "cancer_type": "colon_cancer",
            "biopsy_stage": "C2D15",
        },
    }
)

SAMPLE_ORDER = list(SAMPLE_INFO)

# Preferred exact locations from the validated StarDist/Proseg workflow.
STARDIST_IMAGE_ROOT = (
    TMP_ROOT / "stardist_bigTiff"
)
STARDIST_PROSEG_ROOT = (
    TMP_ROOT / "proseg_stardist_qcclass_tissue_high"
)

print("Samples:", SAMPLE_ORDER)
print("Preferred H&E root:", STARDIST_IMAGE_ROOT)
print("Preferred affine root:", STARDIST_PROSEG_ROOT)


Samples: ['Screen_16_22', 'C2D15_16_22', 'Screen_18_23', 'C2D15_18_23', 'Screen_30_16', 'C2D15_30_16', 'Screen_17_26', 'C2D15_17_26', 'Screen_39_21', 'C2D15_39_21', 'Screen_23_25', 'C2D15_23_25']
Preferred H&E root: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/stardist_bigTiff
Preferred affine root: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_stardist_qcclass_tissue_high


## Edit close-up regions here

The whole-slide outputs do not depend on this dictionary.

Start by running through the ROI-selection-overview section. It writes one
coordinate-grid image per sample. Then add one or more named regions below and
rerun only the ROI-related cells.

The example entries are commented out and do not run until uncommented.


In [3]:
# ---------------------------------------------------------------------
# User-editable close-up ROI definitions
# ---------------------------------------------------------------------
ROI_SPECS = {
     "Screen_16_22": {
    #     "upper_left_FOV1": {
    #         "center_fraction": (2 / 3, 1 / 4),
    #         "size": 2500,
    #     },
         "upper_right_FOV1": {
             "bounds": (10000, 2000, 12000, 4000),
         },
         "upper_middle_FOV1": {
             "bounds": (6000, 4000, 8000, 6000),
         },
         "middle_left_FOV1": {
             "bounds": (4000, 6000, 6000, 8000),
         },
         "lower_left_FOV1": {
             "bounds": (4200, 11000, 6200, 13000),
         },
     },
    "C2D15_16_22": {
         "upper_left_FOV1": {
             "bounds": (3500, 1000, 5500, 3000),
         },
         "upper_middle_FOV1": {
             "bounds": (4000, 2000, 6000, 4000),
         },
         "lower_middle_FOV1": {
             "bounds": (7000, 7500, 9000, 9500),
         },
     },
    #
    #"C2D15_30_16": {
    #     "tumor_edge": {
    #         "center_px": (7200, 3600),
    #         "width": 2400,
    #         "height": 2000,
    #     },
    # },
}

# Coordinate-grid spacing in ROI-selection overviews.
ROI_OVERVIEW_GRID_SPACING_PX = 2000
ROI_OVERVIEW_MAX_SIDE = 3500

# Optional helper-table settings. These do not automatically create ROIs.
ROI_SUGGESTION_WINDOW_SIZE_PX = 2500
ROI_SUGGESTION_STRIDE_PX = 1250
ROI_SUGGESTION_MIN_CELLS = 100
ROI_SUGGESTION_TOP_N = 10

print("Manual ROI samples:", list(ROI_SPECS))


Manual ROI samples: ['Screen_16_22', 'C2D15_16_22']


In [4]:
# ---------------------------------------------------------------------
# Input discovery and coordinate helpers
# ---------------------------------------------------------------------
def discover_merged_handoff() -> Path:
    expected = (
        PIPELINE_ROOT
        / "15_merged_annotation_handoff"
        / MERGED_HANDOFF_FILENAME
    )
    if expected.exists():
        return expected

    candidates = sorted(
        PIPELINE_ROOT.rglob(
            MERGED_HANDOFF_FILENAME
        )
    )
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) == 0:
        raise FileNotFoundError(
            f"Could not locate {MERGED_HANDOFF_FILENAME} "
            f"under {PIPELINE_ROOT}"
        )
    raise RuntimeError(
        "Multiple merged handoff candidates were found:\n"
        + "\n".join(str(path) for path in candidates)
    )


def resolve_he_image(sample: str) -> Path:
    preferred = [
        STARDIST_IMAGE_ROOT
        / f"{sample}_he.tiff",
        STARDIST_IMAGE_ROOT
        / f"{sample}_he.tif",
    ]
    for path in preferred:
        if path.exists():
            return path

    candidates = sorted(
        TMP_ROOT.rglob(
            f"{sample}_he.tif*"
        )
    )
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) == 0:
        raise FileNotFoundError(
            f"No StarDist H&E crop found for {sample}."
        )
    raise RuntimeError(
        f"Multiple H&E candidates found for {sample}:\n"
        + "\n".join(str(path) for path in candidates)
    )


def affine_report_is_valid(report: dict) -> bool:
    return (
        "x_transform" in report
        and "y_transform" in report
        and len(report["x_transform"]) == 3
        and len(report["y_transform"]) == 3
    )


def resolve_affine_json(sample: str) -> Path:
    preferred = [
        STARDIST_PROSEG_ROOT
        / sample
        / f"{sample}_mask_to_proseg_affine.json",
    ]
    for path in preferred:
        if path.exists():
            report = json.loads(path.read_text())
            if affine_report_is_valid(report):
                return path

    candidates = sorted(
        TMP_ROOT.rglob(
            f"{sample}_mask_to_proseg_affine.json"
        )
    )
    valid = []
    for path in candidates:
        try:
            report = json.loads(path.read_text())
            if affine_report_is_valid(report):
                valid.append(path)
        except Exception:
            continue

    if len(valid) == 1:
        return valid[0]
    if len(valid) == 0:
        raise FileNotFoundError(
            f"No valid mask-to-Proseg affine JSON found for {sample}."
        )
    raise RuntimeError(
        f"Multiple valid affine candidates found for {sample}:\n"
        + "\n".join(str(path) for path in valid)
    )


def proseg_um_to_image_pixels(
    xy_um: np.ndarray,
    affine_report: dict,
) -> np.ndarray:
    """
    Convert Proseg x/y micron coordinates to StarDist-image x/y pixels.

    The stored affine maps image pixels to Proseg coordinates:

        proseg_x = a * image_x + b * image_y + c
        proseg_y = d * image_x + e * image_y + f

    This function applies its inverse.
    """
    x = np.asarray(
        affine_report["x_transform"],
        dtype=np.float64,
    )
    y = np.asarray(
        affine_report["y_transform"],
        dtype=np.float64,
    )

    linear = np.array(
        [
            [x[0], x[1]],
            [y[0], y[1]],
        ],
        dtype=np.float64,
    )
    offset = np.array(
        [x[2], y[2]],
        dtype=np.float64,
    )

    determinant = float(
        np.linalg.det(linear)
    )
    if abs(determinant) < 1e-12:
        raise ValueError(
            f"Affine linear component is singular: determinant={determinant}"
        )

    inverse = np.linalg.inv(linear)
    image_xy = (
        np.asarray(xy_um, dtype=np.float64)
        - offset
    ) @ inverse.T

    return image_xy.astype(
        np.float32,
        copy=False,
    )


MERGED_HANDOFF_PATH = discover_merged_handoff()

print("Merged handoff:", MERGED_HANDOFF_PATH)


Merged handoff: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/15_merged_annotation_handoff/VisiumHD_12samples_ResolVI_corrected_preliminary_annotation.zarr


In [5]:
# ---------------------------------------------------------------------
# Load Step 18 annotations and merged spatial coordinates lazily
# ---------------------------------------------------------------------
def materialize_frame(value) -> pd.DataFrame:
    if hasattr(value, "to_memory"):
        value = value.to_memory()
    if hasattr(value, "compute"):
        value = value.compute()
    return pd.DataFrame(value).copy()


def materialize_array(value) -> np.ndarray:
    if hasattr(value, "to_memory"):
        value = value.to_memory()
    if hasattr(value, "compute"):
        value = value.compute()
    return np.asarray(value)


if not ANNOTATION_HANDOFF_PATH.exists():
    raise FileNotFoundError(
        ANNOTATION_HANDOFF_PATH
    )
if not PALETTE_JSON_PATH.exists():
    raise FileNotFoundError(
        PALETTE_JSON_PATH
    )

annotations = pd.read_parquet(
    ANNOTATION_HANDOFF_PATH
)
if "cell_id" not in annotations.columns:
    raise KeyError(
        "Step 18 handoff lacks a cell_id column."
    )
annotations = annotations.set_index(
    "cell_id",
    drop=True,
)
annotations.index = annotations.index.astype(str)
annotations.index.name = "cell_id"

for required in [
    "sample",
    SPECIFIC_COLUMN,
    BROADER_COLUMN,
]:
    if required not in annotations:
        raise KeyError(
            f"Step 18 handoff lacks {required!r}."
        )

palette = json.loads(
    PALETTE_JSON_PATH.read_text()
)
palette = {
    str(label): str(color)
    for label, color in palette.items()
}

if PALETTE_CSV_PATH.exists():
    palette_table = pd.read_csv(
        PALETTE_CSV_PATH
    )
    palette_order = (
        palette_table
        .sort_values("order")[
            "cell_type"
        ]
        .astype(str)
        .tolist()
    )
else:
    palette_order = list(palette)

if not hasattr(ad.experimental, "read_lazy"):
    raise RuntimeError(
        "This notebook requires anndata.experimental.read_lazy() "
        "so the dense merged expression matrix is not materialized."
    )

merged_lazy = ad.experimental.read_lazy(
    str(MERGED_HANDOFF_PATH)
)
merged_obs = materialize_frame(
    merged_lazy.obs
)
merged_obs.index = merged_obs.index.astype(str)

if "spatial" not in merged_lazy.obsm:
    raise KeyError(
        "Merged handoff lacks obsm['spatial']."
    )

merged_spatial = materialize_array(
    merged_lazy.obsm["spatial"]
).astype(np.float32, copy=False)

if merged_spatial.shape != (
    len(merged_obs),
    2,
):
    raise ValueError(
        f"Unexpected spatial shape {merged_spatial.shape}; "
        f"expected {(len(merged_obs), 2)}."
    )

missing_annotations = (
    merged_obs.index.difference(
        annotations.index
    )
)
if len(missing_annotations):
    raise RuntimeError(
        f"{len(missing_annotations):,} merged cells are absent "
        "from the Step 18 annotation handoff."
    )

annotations = annotations.reindex(
    merged_obs.index
)

if annotations[
    [
        SPECIFIC_COLUMN,
        BROADER_COLUMN,
    ]
].isna().any().any():
    raise RuntimeError(
        "Step 18 annotations contain missing values after alignment."
    )

# Store spatial coordinates separately while retaining exact merged order.
annotations["proseg_x_um"] = (
    merged_spatial[:, 0]
)
annotations["proseg_y_um"] = (
    merged_spatial[:, 1]
)

print("Aligned cells:", f"{len(annotations):,}")
print("Samples:", annotations["sample"].astype(str).nunique())
print("Palette labels:", len(palette))


/home/domino/reny28/Projects/Visium_projects/TBIO-8057/env/gpu_env/.venv/lib/python3.13/site-packages/anndata/_core/xarray.py:32: UserWarning: Did not read zarr as consolidated. Consider consolidating your metadata.
  return func(*args, **kwargs)


Aligned cells: 862,427
Samples: 12
Palette labels: 20


In [6]:
# ---------------------------------------------------------------------
# Per-sample image-space coordinate preparation and validation
# ---------------------------------------------------------------------
sample_context = {}
preflight_rows = []

for sample in SAMPLE_ORDER:
    image_path = resolve_he_image(sample)
    affine_path = resolve_affine_json(sample)
    affine_report = json.loads(
        affine_path.read_text()
    )

    with Image.open(image_path) as image:
        width, height = image.size

    sample_mask = (
        annotations["sample"]
        .astype(str)
        .eq(sample)
        .to_numpy()
    )
    sample_positions = np.flatnonzero(
        sample_mask
    )

    xy_um = annotations.iloc[
        sample_positions
    ][
        [
            "proseg_x_um",
            "proseg_y_um",
        ]
    ].to_numpy(dtype=np.float32)

    image_xy = proseg_um_to_image_pixels(
        xy_um,
        affine_report,
    )

    in_bounds = (
        (image_xy[:, 0] >= 0)
        & (image_xy[:, 0] < width)
        & (image_xy[:, 1] >= 0)
        & (image_xy[:, 1] < height)
    )
    fraction_in_bounds = float(
        in_bounds.mean()
    )

    if fraction_in_bounds < MIN_CELLS_IN_IMAGE_FRACTION:
        raise RuntimeError(
            f"{sample}: only {fraction_in_bounds:.3%} of cell "
            "centroids fall inside the H&E crop. Recheck the image/affine pair."
        )

    sample_context[sample] = {
        "sample": sample,
        "image_path": image_path,
        "affine_path": affine_path,
        "affine_report": affine_report,
        "width": int(width),
        "height": int(height),
        "global_positions": sample_positions,
        "image_xy": image_xy,
        "in_bounds": in_bounds,
    }

    preflight_rows.append(
        {
            "sample": sample,
            "image_path": str(image_path),
            "affine_path": str(affine_path),
            "image_width_px": int(width),
            "image_height_px": int(height),
            "n_cells": int(len(image_xy)),
            "n_cells_in_bounds": int(
                in_bounds.sum()
            ),
            "fraction_cells_in_bounds": (
                fraction_in_bounds
            ),
            "specific_labels": int(
                annotations.iloc[
                    sample_positions
                ][
                    SPECIFIC_COLUMN
                ].nunique()
            ),
            "broader_labels": int(
                annotations.iloc[
                    sample_positions
                ][
                    BROADER_COLUMN
                ].nunique()
            ),
        }
    )

preflight = pd.DataFrame(
    preflight_rows
)
preflight.to_csv(
    OUTPUT_ROOT
    / "HE_overlay_preflight.csv",
    index=False,
)

display(preflight)


,sample,image_path,affine_path,image_width_px,image_height_px,n_cells,n_cells_in_bounds,fraction_cells_in_bounds,specific_labels,broader_labels
0,Screen_16_22,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,12640,15280,70438,70438,1.0,14,11
1,C2D15_16_22,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,12159,10988,76633,76633,1.0,15,12
2,Screen_18_23,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,22656,13398,72386,72386,1.0,13,10
3,C2D15_18_23,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,17655,18877,18594,18594,1.0,14,11
4,Screen_30_16,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,22670,22670,66977,66977,1.0,15,12
5,C2D15_30_16,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,22566,22568,351799,351799,1.0,11,9
6,Screen_17_26,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,22545,22513,47896,47896,1.0,13,11
7,C2D15_17_26,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,22859,22497,87913,87913,1.0,12,10
8,Screen_39_21,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,18825,19542,22950,22950,1.0,13,11
9,C2D15_39_21,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,10786,20588,10822,10822,1.0,12,10


In [7]:
# ---------------------------------------------------------------------
# Image and plotting helpers
# ---------------------------------------------------------------------
def prepare_he_image(
    image: Image.Image,
) -> Image.Image:
    result = image.convert("RGB")

    if HE_BRIGHTNESS != 1.0:
        result = ImageEnhance.Brightness(
            result
        ).enhance(HE_BRIGHTNESS)

    if HE_CONTRAST != 1.0:
        result = ImageEnhance.Contrast(
            result
        ).enhance(HE_CONTRAST)

    if HE_COLOR_SATURATION != 1.0:
        result = ImageEnhance.Color(
            result
        ).enhance(HE_COLOR_SATURATION)

    return result


def figure_size_for_image(
    width: int,
    height: int,
    max_inches: float,
    min_inches: float = 5.0,
) -> tuple[float, float]:
    if width <= 0 or height <= 0:
        raise ValueError(
            f"Invalid image size {(width, height)}."
        )

    if width >= height:
        return (
            max_inches,
            max(
                min_inches,
                max_inches
                * height
                / width,
            ),
        )

    return (
        max(
            min_inches,
            max_inches
            * width
            / height,
        ),
        max_inches,
    )


def labels_present_in_order(
    values: pd.Series,
) -> list[str]:
    present = set(
        values.astype(str)
    )
    ordered = [
        label
        for label in palette_order
        if label in present
    ]
    extras = sorted(
        present.difference(ordered)
    )

    missing_colors = [
        label
        for label in extras
        if label not in palette
    ]
    if missing_colors:
        raise KeyError(
            "Palette is missing labels: "
            f"{missing_colors}"
        )

    return [
        *ordered,
        *extras,
    ]


def legend_handles(
    labels: list[str],
) -> list[Line2D]:
    return [
        Line2D(
            [],
            [],
            marker="o",
            linestyle="",
            markersize=6,
            markerfacecolor=palette[label],
            markeredgecolor="none",
            alpha=0.90,
            label=label,
        )
        for label in labels
    ]


def scatter_annotation(
    ax,
    coordinates: np.ndarray,
    labels: pd.Series,
    *,
    point_size: float,
    title: str,
):
    labels = labels.astype(str)
    counts = labels.value_counts()

    if DRAW_ABUNDANT_FIRST:
        draw_order = (
            counts.sort_values(
                ascending=False
            )
            .index.astype(str)
            .tolist()
        )
    else:
        draw_order = labels_present_in_order(
            labels
        )

    label_array = labels.to_numpy()

    for label in draw_order:
        mask = label_array == label
        ax.scatter(
            coordinates[mask, 0],
            coordinates[mask, 1],
            s=point_size,
            color=palette[label],
            alpha=CELL_ALPHA,
            linewidths=0,
            rasterized=True,
        )

    ax.set_title(
        title,
        fontsize=14,
    )
    ax.set_aspect(
        "equal",
        adjustable="box",
    )
    ax.axis("off")

    return labels_present_in_order(
        labels
    )


def sample_annotation_data(
    sample: str,
    annotation_column: str,
):
    context = sample_context[sample]
    positions = context["global_positions"]
    values = annotations.iloc[
        positions
    ][annotation_column].copy()

    return (
        context,
        context["image_xy"],
        values,
    )


def preview_image_and_scale(
    image_path: Path,
    max_side: int,
):
    with Image.open(image_path) as image:
        full_width, full_height = image.size
        preview = prepare_he_image(
            image
        )
        preview.thumbnail(
            (max_side, max_side),
            Image.Resampling.LANCZOS,
        )

    scale_x = preview.width / full_width
    scale_y = preview.height / full_height

    return (
        np.asarray(preview),
        scale_x,
        scale_y,
        full_width,
        full_height,
    )


def output_is_reusable(path: Path) -> bool:
    return (
        path.exists()
        and path.stat().st_size > 0
        and not OVERWRITE
    )


In [8]:
# ---------------------------------------------------------------------
# Whole-capture-area H&E overlay writers
# ---------------------------------------------------------------------
WHOLE_ROOT = (
    OUTPUT_ROOT / "whole_slide"
)
WHOLE_SPECIFIC_ROOT = (
    WHOLE_ROOT / "specific"
)
WHOLE_BROADER_ROOT = (
    WHOLE_ROOT / "broader"
)
WHOLE_COMPARISON_ROOT = (
    WHOLE_ROOT / "specific_vs_broader"
)

for path in [
    WHOLE_SPECIFIC_ROOT,
    WHOLE_BROADER_ROOT,
    WHOLE_COMPARISON_ROOT,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )


def save_whole_single(
    sample: str,
    annotation_column: str,
    display_name: str,
    output_path: Path,
):
    if output_is_reusable(output_path):
        print("Reusing:", output_path)
        return

    context, coordinates, labels = (
        sample_annotation_data(
            sample,
            annotation_column,
        )
    )

    (
        preview,
        scale_x,
        scale_y,
        width,
        height,
    ) = preview_image_and_scale(
        context["image_path"],
        WHOLE_PREVIEW_MAX_SIDE,
    )

    preview_coordinates = (
        coordinates
        * np.array(
            [scale_x, scale_y],
            dtype=np.float32,
        )
    )

    fig, ax = plt.subplots(
        figsize=figure_size_for_image(
            width,
            height,
            WHOLE_FIGURE_MAX_INCHES,
        )
    )
    ax.imshow(
        preview,
        extent=(0, width, height, 0),
        interpolation="none",
    )

    # Plot in native image-pixel coordinates. The preview is stretched back to
    # the native extent so the affine-derived centroids remain directly usable.
    present = scatter_annotation(
        ax,
        coordinates,
        labels,
        point_size=WHOLE_POINT_SIZE,
        title=(
            f"{sample}\n{display_name} over H&E"
        ),
    )

    ax.set_xlim(0, width)
    ax.set_ylim(height, 0)

    if SHOW_LEGEND:
        ax.legend(
            handles=legend_handles(
                present
            ),
            loc="center left",
            bbox_to_anchor=(1.01, 0.5),
            frameon=False,
            fontsize=LEGEND_FONT_SIZE,
            ncol=LEGEND_COLUMNS_WHOLE,
        )

    fig.tight_layout()
    fig.savefig(
        output_path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
        pad_inches=0.03,
    )
    plt.close(fig)


def save_whole_comparison(
    sample: str,
    output_path: Path,
):
    if output_is_reusable(output_path):
        print("Reusing:", output_path)
        return

    context = sample_context[sample]
    positions = context["global_positions"]
    coordinates = context["image_xy"]
    width = context["width"]
    height = context["height"]

    specific = annotations.iloc[
        positions
    ][SPECIFIC_COLUMN]
    broader = annotations.iloc[
        positions
    ][BROADER_COLUMN]

    preview, _, _, _, _ = (
        preview_image_and_scale(
            context["image_path"],
            WHOLE_PREVIEW_MAX_SIDE,
        )
    )

    one_width, one_height = figure_size_for_image(
        width,
        height,
        WHOLE_FIGURE_MAX_INCHES,
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(
            2 * one_width,
            one_height,
        ),
    )

    present_specific = scatter_annotation(
        axes[0],
        coordinates,
        specific,
        point_size=WHOLE_POINT_SIZE,
        title="Specific cell-type annotation",
    )
    present_broader = scatter_annotation(
        axes[1],
        coordinates,
        broader,
        point_size=WHOLE_POINT_SIZE,
        title="Broader cell-type annotation",
    )

    for ax in axes:
        ax.imshow(
            preview,
            extent=(0, width, height, 0),
            interpolation="none",
            zorder=-10,
        )
        ax.set_xlim(0, width)
        ax.set_ylim(height, 0)

    present_union = [
        label
        for label in palette_order
        if label
        in set(
            [
                *present_specific,
                *present_broader,
            ]
        )
    ]

    fig.suptitle(
        f"{sample}: harmonized annotations over H&E",
        fontsize=16,
    )

    if SHOW_LEGEND:
        fig.legend(
            handles=legend_handles(
                present_union
            ),
            loc="lower center",
            bbox_to_anchor=(0.5, -0.01),
            frameon=False,
            fontsize=LEGEND_FONT_SIZE,
            ncol=min(
                LEGEND_COLUMNS_COMPARISON,
                max(1, len(present_union)),
            ),
        )

    fig.tight_layout(
        rect=(0, 0.08, 1, 0.96)
    )
    fig.savefig(
        output_path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
        pad_inches=0.03,
    )
    plt.close(fig)


In [9]:
# ---------------------------------------------------------------------
# Generate whole-slide/capture-area views for all samples
# ---------------------------------------------------------------------
whole_manifest_rows = []

if RUN_WHOLE_SLIDE_PLOTS:
    for sample in SAMPLE_ORDER:
        print("\nWhole H&E overlays:", sample)

        specific_path = (
            WHOLE_SPECIFIC_ROOT
            / f"{sample}_Specific_annotation_over_HE.png"
        )
        broader_path = (
            WHOLE_BROADER_ROOT
            / f"{sample}_Broader_annotation_over_HE.png"
        )
        comparison_path = (
            WHOLE_COMPARISON_ROOT
            / f"{sample}_Specific_vs_Broader_over_HE.png"
        )

        if MAKE_SEPARATE_SPECIFIC:
            save_whole_single(
                sample,
                SPECIFIC_COLUMN,
                "Specific cell-type annotation",
                specific_path,
            )

        if MAKE_SEPARATE_BROADER:
            save_whole_single(
                sample,
                BROADER_COLUMN,
                "Broader cell-type annotation",
                broader_path,
            )

        if MAKE_SIDE_BY_SIDE:
            save_whole_comparison(
                sample,
                comparison_path,
            )

        whole_manifest_rows.append(
            {
                "sample": sample,
                "specific": str(specific_path),
                "broader": str(broader_path),
                "comparison": str(comparison_path),
            }
        )

whole_manifest = pd.DataFrame(
    whole_manifest_rows
)
whole_manifest.to_csv(
    OUTPUT_ROOT
    / "whole_slide_HE_overlay_manifest.csv",
    index=False,
)

display(whole_manifest)



Whole H&E overlays: Screen_16_22
Reusing: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/19_harmonized_annotations_over_HE/whole_slide/specific/Screen_16_22_Specific_annotation_over_HE.png
Reusing: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/19_harmonized_annotations_over_HE/whole_slide/broader/Screen_16_22_Broader_annotation_over_HE.png
Reusing: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/19_harmonized_annotations_over_HE/whole_slide/specific_vs_broader/Screen_16_22_Specific_vs_Broader_over_HE.png

Whole H&E overlays: C2D15_16_22
Reusing: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/19_harmonized_annotations_over_HE/whole_slide/specific/C2D15_16_22_Specific_annotation_over_HE.png
Reusing: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_reso

,sample,specific,broader,comparison
0,Screen_16_22,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
1,C2D15_16_22,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
2,Screen_18_23,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
3,C2D15_18_23,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
4,Screen_30_16,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
5,C2D15_30_16,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
6,Screen_17_26,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
7,C2D15_17_26,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
8,Screen_39_21,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
9,C2D15_39_21,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...


## ROI planning tools

The next cells do three things:

1. resolve manual ROI specifications;
2. create coordinate-grid overview images;
3. provide an optional ranking of candidate windows by:
   - cell density;
   - annotation diversity;
   - number/fraction of cells whose Specific and Broader labels differ.

The candidate table is only a planning aid. It does not automatically modify
`ROI_SPECS`.


In [10]:
# ---------------------------------------------------------------------
# ROI resolution and optional candidate-window suggestions
# ---------------------------------------------------------------------
def clip_roi(
    left: float,
    upper: float,
    right: float,
    lower: float,
    width: int,
    height: int,
) -> tuple[int, int, int, int]:
    left = int(
        np.clip(
            round(left),
            0,
            max(width - 1, 0),
        )
    )
    upper = int(
        np.clip(
            round(upper),
            0,
            max(height - 1, 0),
        )
    )
    right = int(
        np.clip(
            round(right),
            left + 1,
            width,
        )
    )
    lower = int(
        np.clip(
            round(lower),
            upper + 1,
            height,
        )
    )
    return (
        left,
        upper,
        right,
        lower,
    )


def resolve_roi_spec(
    spec,
    width: int,
    height: int,
) -> tuple[int, int, int, int]:
    if (
        isinstance(spec, (list, tuple))
        and len(spec) == 4
    ):
        return clip_roi(
            *spec,
            width,
            height,
        )

    if not isinstance(spec, dict):
        raise TypeError(
            f"ROI specification must be a dict or four-value tuple: {spec}"
        )

    if "bounds" in spec:
        bounds = spec["bounds"]
        if len(bounds) != 4:
            raise ValueError(
                f"ROI bounds must have four values: {bounds}"
            )
        return clip_roi(
            *bounds,
            width,
            height,
        )

    roi_width = int(
        spec.get(
            "width",
            spec.get(
                "size",
                2500,
            ),
        )
    )
    roi_height = int(
        spec.get(
            "height",
            spec.get(
                "size",
                roi_width,
            ),
        )
    )

    if "center_fraction" in spec:
        x_fraction, y_fraction = (
            spec["center_fraction"]
        )
        center_x = float(x_fraction) * width
        center_y = float(y_fraction) * height
    elif "center_px" in spec:
        center_x, center_y = (
            spec["center_px"]
        )
    else:
        raise KeyError(
            "ROI dict must contain bounds, center_fraction, or center_px."
        )

    left = center_x - roi_width / 2
    upper = center_y - roi_height / 2
    right = left + roi_width
    lower = upper + roi_height

    # Shift the ROI when necessary so its requested size remains inside.
    if left < 0:
        right -= left
        left = 0
    if upper < 0:
        lower -= upper
        upper = 0
    if right > width:
        left -= right - width
        right = width
    if lower > height:
        upper -= lower - height
        lower = height

    return clip_roi(
        left,
        upper,
        right,
        lower,
        width,
        height,
    )


def resolved_rois_for_sample(
    sample: str,
) -> OrderedDict:
    context = sample_context[sample]
    result = OrderedDict()

    for name, spec in ROI_SPECS.get(
        sample,
        {},
    ).items():
        result[str(name)] = (
            resolve_roi_spec(
                spec,
                context["width"],
                context["height"],
            )
        )

    return result


def suggest_roi_windows(
    sample: str,
    *,
    top_n: int = ROI_SUGGESTION_TOP_N,
) -> pd.DataFrame:
    """
    Rank fixed windows for ROI planning.

    High disagreement_count highlights where the Specific and Broader schemes
    differ. High diversity_score highlights locally mixed cell-type regions.
    """
    context = sample_context[sample]
    positions = context["global_positions"]
    coordinates = context["image_xy"]
    specific = annotations.iloc[
        positions
    ][SPECIFIC_COLUMN].astype(str).to_numpy()
    broader = annotations.iloc[
        positions
    ][BROADER_COLUMN].astype(str).to_numpy()

    width = context["width"]
    height = context["height"]
    window = int(
        ROI_SUGGESTION_WINDOW_SIZE_PX
    )
    stride = int(
        ROI_SUGGESTION_STRIDE_PX
    )

    rows = []
    for upper in range(
        0,
        max(height - window + 1, 1),
        stride,
    ):
        for left in range(
            0,
            max(width - window + 1, 1),
            stride,
        ):
            right = min(
                left + window,
                width,
            )
            lower = min(
                upper + window,
                height,
            )

            mask = (
                (coordinates[:, 0] >= left)
                & (coordinates[:, 0] < right)
                & (coordinates[:, 1] >= upper)
                & (coordinates[:, 1] < lower)
            )
            n_cells = int(mask.sum())
            if n_cells < ROI_SUGGESTION_MIN_CELLS:
                continue

            disagreement = (
                specific[mask]
                != broader[mask]
            )
            n_specific = int(
                pd.Series(
                    specific[mask]
                ).nunique()
            )
            n_broader = int(
                pd.Series(
                    broader[mask]
                ).nunique()
            )

            rows.append(
                {
                    "sample": sample,
                    "left": left,
                    "upper": upper,
                    "right": right,
                    "lower": lower,
                    "center_x": (
                        left + right
                    ) / 2,
                    "center_y": (
                        upper + lower
                    ) / 2,
                    "n_cells": n_cells,
                    "n_specific_labels": n_specific,
                    "n_broader_labels": n_broader,
                    "diversity_score": (
                        n_specific
                        + n_broader
                    ),
                    "disagreement_count": int(
                        disagreement.sum()
                    ),
                    "disagreement_fraction": float(
                        disagreement.mean()
                    ),
                }
            )

    table = pd.DataFrame(rows)
    if table.empty:
        return table

    table = table.sort_values(
        [
            "disagreement_count",
            "diversity_score",
            "n_cells",
        ],
        ascending=False,
    ).reset_index(drop=True)

    return table.head(top_n)


In [11]:
# ---------------------------------------------------------------------
# Coordinate-grid ROI-selection overviews
# ---------------------------------------------------------------------
ROI_OVERVIEW_ROOT = (
    OUTPUT_ROOT
    / "ROI_selection_overviews"
)
ROI_OVERVIEW_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


def save_roi_selection_overview(
    sample: str,
    output_path: Path,
):
    context = sample_context[sample]
    width = context["width"]
    height = context["height"]

    (
        preview,
        _,
        _,
        _,
        _,
    ) = preview_image_and_scale(
        context["image_path"],
        ROI_OVERVIEW_MAX_SIDE,
    )

    fig, ax = plt.subplots(
        figsize=figure_size_for_image(
            width,
            height,
            12.0,
        )
    )
    ax.imshow(
        preview,
        extent=(0, width, height, 0),
        interpolation="none",
    )

    spacing = int(
        ROI_OVERVIEW_GRID_SPACING_PX
    )
    ax.set_xticks(
        np.arange(
            0,
            width + 1,
            spacing,
        )
    )
    ax.set_yticks(
        np.arange(
            0,
            height + 1,
            spacing,
        )
    )
    ax.grid(
        color="white",
        alpha=0.60,
        linewidth=0.7,
    )
    ax.tick_params(
        labelsize=8,
    )

    for roi_name, (
        left,
        upper,
        right,
        lower,
    ) in resolved_rois_for_sample(
        sample
    ).items():
        rectangle = Rectangle(
            (left, upper),
            right - left,
            lower - upper,
            fill=False,
            linewidth=2.0,
            edgecolor="#00FFFF",
        )
        ax.add_patch(rectangle)
        ax.text(
            left,
            max(0, upper - 60),
            roi_name,
            color="#00FFFF",
            fontsize=9,
            weight="bold",
            bbox={
                "facecolor": "black",
                "alpha": 0.55,
                "edgecolor": "none",
                "pad": 2,
            },
        )

    ax.set_xlim(0, width)
    ax.set_ylim(height, 0)
    ax.set_aspect(
        "equal",
        adjustable="box",
    )
    ax.set_xlabel(
        "H&E crop x coordinate (pixels)"
    )
    ax.set_ylabel(
        "H&E crop y coordinate (pixels)"
    )
    ax.set_title(
        f"{sample}: ROI-selection coordinate grid"
    )

    fig.tight_layout()
    fig.savefig(
        output_path,
        dpi=220,
        bbox_inches="tight",
    )
    plt.close(fig)


roi_dimension_rows = []

if RUN_ROI_SELECTION_OVERVIEWS:
    for sample in SAMPLE_ORDER:
        path = (
            ROI_OVERVIEW_ROOT
            / f"{sample}_ROI_selection_grid.png"
        )
        save_roi_selection_overview(
            sample,
            path,
        )

        context = sample_context[sample]
        roi_dimension_rows.append(
            {
                "sample": sample,
                "image_width_px": (
                    context["width"]
                ),
                "image_height_px": (
                    context["height"]
                ),
                "overview": str(path),
                "n_manual_rois": len(
                    resolved_rois_for_sample(
                        sample
                    )
                ),
            }
        )

roi_dimensions = pd.DataFrame(
    roi_dimension_rows
)
roi_dimensions.to_csv(
    OUTPUT_ROOT
    / "ROI_image_dimensions_and_overviews.csv",
    index=False,
)

display(roi_dimensions)


,sample,image_width_px,image_height_px,overview,n_manual_rois
0,Screen_16_22,12640,15280,/host_root/nethome/reny28/Projects/Visium_proj...,4
1,C2D15_16_22,12159,10988,/host_root/nethome/reny28/Projects/Visium_proj...,3
2,Screen_18_23,22656,13398,/host_root/nethome/reny28/Projects/Visium_proj...,0
3,C2D15_18_23,17655,18877,/host_root/nethome/reny28/Projects/Visium_proj...,0
4,Screen_30_16,22670,22670,/host_root/nethome/reny28/Projects/Visium_proj...,0
5,C2D15_30_16,22566,22568,/host_root/nethome/reny28/Projects/Visium_proj...,0
6,Screen_17_26,22545,22513,/host_root/nethome/reny28/Projects/Visium_proj...,0
7,C2D15_17_26,22859,22497,/host_root/nethome/reny28/Projects/Visium_proj...,0
8,Screen_39_21,18825,19542,/host_root/nethome/reny28/Projects/Visium_proj...,0
9,C2D15_39_21,10786,20588,/host_root/nethome/reny28/Projects/Visium_proj...,0


## Optional: inspect candidate ROI windows

Run the following example for a sample of interest:

```python
display(
    suggest_roi_windows("Screen_16_22")
)
```

To convert a candidate row into `ROI_SPECS`:

```python
ROI_SPECS["Screen_16_22"] = {
    "candidate_1": {
        "bounds": (
            int(row["left"]),
            int(row["upper"]),
            int(row["right"]),
            int(row["lower"]),
        )
    }
}
```

Then rerun:

```text
the ROI-selection-overview cell
the close-up-generation cell
```

The whole-slide figures do not need to be rerun.


In [12]:
# ---------------------------------------------------------------------
# Close-up H&E overlay writers
# ---------------------------------------------------------------------
CLOSEUP_ROOT = (
    OUTPUT_ROOT / "closeups"
)
CLOSEUP_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


def cells_in_roi(
    sample: str,
    bounds: tuple[int, int, int, int],
):
    context = sample_context[sample]
    positions = context["global_positions"]
    coordinates = context["image_xy"]

    left, upper, right, lower = bounds
    mask = (
        (coordinates[:, 0] >= left)
        & (coordinates[:, 0] < right)
        & (coordinates[:, 1] >= upper)
        & (coordinates[:, 1] < lower)
    )

    return (
        positions[mask],
        coordinates[mask],
    )


def load_native_roi_image(
    image_path: Path,
    bounds: tuple[int, int, int, int],
) -> np.ndarray:
    with Image.open(image_path) as image:
        crop = image.crop(
            bounds
        )
        crop = prepare_he_image(
            crop
        )
        return np.asarray(crop)


def save_roi_single(
    sample: str,
    roi_name: str,
    bounds: tuple[int, int, int, int],
    annotation_column: str,
    display_name: str,
    output_path: Path,
):
    if output_is_reusable(output_path):
        print("Reusing:", output_path)
        return

    context = sample_context[sample]
    positions, coordinates = cells_in_roi(
        sample,
        bounds,
    )
    labels = annotations.iloc[
        positions
    ][annotation_column]

    left, upper, right, lower = bounds
    crop = load_native_roi_image(
        context["image_path"],
        bounds,
    )

    fig, ax = plt.subplots(
        figsize=figure_size_for_image(
            right - left,
            lower - upper,
            ROI_FIGURE_MAX_INCHES,
        )
    )
    ax.imshow(
        crop,
        extent=(
            left,
            right,
            lower,
            upper,
        ),
        interpolation="none",
    )

    present = scatter_annotation(
        ax,
        coordinates,
        labels,
        point_size=ROI_POINT_SIZE,
        title=(
            f"{sample} | {roi_name}\n"
            f"{display_name} over H&E | "
            f"{len(positions):,} cells"
        ),
    )
    ax.set_xlim(left, right)
    ax.set_ylim(lower, upper)

    if SHOW_LEGEND:
        ax.legend(
            handles=legend_handles(
                present
            ),
            loc="center left",
            bbox_to_anchor=(1.01, 0.5),
            frameon=False,
            fontsize=LEGEND_FONT_SIZE,
        )

    fig.tight_layout()
    fig.savefig(
        output_path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
        pad_inches=0.03,
    )
    plt.close(fig)


def save_roi_comparison(
    sample: str,
    roi_name: str,
    bounds: tuple[int, int, int, int],
    output_path: Path,
):
    if output_is_reusable(output_path):
        print("Reusing:", output_path)
        return

    context = sample_context[sample]
    positions, coordinates = cells_in_roi(
        sample,
        bounds,
    )
    specific = annotations.iloc[
        positions
    ][SPECIFIC_COLUMN]
    broader = annotations.iloc[
        positions
    ][BROADER_COLUMN]

    left, upper, right, lower = bounds
    crop = load_native_roi_image(
        context["image_path"],
        bounds,
    )

    one_width, one_height = (
        figure_size_for_image(
            right - left,
            lower - upper,
            ROI_FIGURE_MAX_INCHES,
        )
    )

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(
            2 * one_width,
            one_height,
        ),
    )

    for ax in axes:
        ax.imshow(
            crop,
            extent=(
                left,
                right,
                lower,
                upper,
            ),
            interpolation="none",
            zorder=-10,
        )

    present_specific = scatter_annotation(
        axes[0],
        coordinates,
        specific,
        point_size=ROI_POINT_SIZE,
        title="Specific annotation",
    )
    present_broader = scatter_annotation(
        axes[1],
        coordinates,
        broader,
        point_size=ROI_POINT_SIZE,
        title="Broader annotation",
    )

    for ax in axes:
        ax.set_xlim(left, right)
        ax.set_ylim(lower, upper)

    present_union = [
        label
        for label in palette_order
        if label in set(
            [
                *present_specific,
                *present_broader,
            ]
        )
    ]

    fig.suptitle(
        f"{sample} | {roi_name} | {len(positions):,} cells",
        fontsize=16,
    )

    if SHOW_LEGEND:
        fig.legend(
            handles=legend_handles(
                present_union
            ),
            loc="lower center",
            bbox_to_anchor=(0.5, -0.01),
            frameon=False,
            fontsize=LEGEND_FONT_SIZE,
            ncol=min(
                LEGEND_COLUMNS_COMPARISON,
                max(1, len(present_union)),
            ),
        )

    fig.tight_layout(
        rect=(0, 0.08, 1, 0.95)
    )
    fig.savefig(
        output_path,
        dpi=PLOT_DPI,
        bbox_inches="tight",
        pad_inches=0.03,
    )
    plt.close(fig)


In [13]:
# ---------------------------------------------------------------------
# Generate all user-defined close-up views
# ---------------------------------------------------------------------
closeup_manifest_rows = []

if RUN_CLOSEUP_PLOTS:
    for sample in SAMPLE_ORDER:
        resolved = (
            resolved_rois_for_sample(
                sample
            )
        )
        if not resolved:
            continue

        for roi_name, bounds in resolved.items():
            print(
                f"Close-up: {sample} | {roi_name} | {bounds}"
            )

            roi_root = (
                CLOSEUP_ROOT
                / sample
                / roi_name
            )
            roi_root.mkdir(
                parents=True,
                exist_ok=True,
            )

            specific_path = (
                roi_root
                / (
                    f"{sample}_{roi_name}_"
                    "Specific_annotation_over_HE.png"
                )
            )
            broader_path = (
                roi_root
                / (
                    f"{sample}_{roi_name}_"
                    "Broader_annotation_over_HE.png"
                )
            )
            comparison_path = (
                roi_root
                / (
                    f"{sample}_{roi_name}_"
                    "Specific_vs_Broader_over_HE.png"
                )
            )

            if MAKE_SEPARATE_SPECIFIC:
                save_roi_single(
                    sample,
                    roi_name,
                    bounds,
                    SPECIFIC_COLUMN,
                    "Specific cell-type annotation",
                    specific_path,
                )

            if MAKE_SEPARATE_BROADER:
                save_roi_single(
                    sample,
                    roi_name,
                    bounds,
                    BROADER_COLUMN,
                    "Broader cell-type annotation",
                    broader_path,
                )

            if MAKE_SIDE_BY_SIDE:
                save_roi_comparison(
                    sample,
                    roi_name,
                    bounds,
                    comparison_path,
                )

            positions, _ = cells_in_roi(
                sample,
                bounds,
            )

            closeup_manifest_rows.append(
                {
                    "sample": sample,
                    "roi_name": roi_name,
                    "left": bounds[0],
                    "upper": bounds[1],
                    "right": bounds[2],
                    "lower": bounds[3],
                    "n_cells": int(
                        len(positions)
                    ),
                    "specific": str(
                        specific_path
                    ),
                    "broader": str(
                        broader_path
                    ),
                    "comparison": str(
                        comparison_path
                    ),
                }
            )

closeup_manifest = pd.DataFrame(
    closeup_manifest_rows
)
closeup_manifest.to_csv(
    OUTPUT_ROOT
    / "closeup_HE_overlay_manifest.csv",
    index=False,
)

display(closeup_manifest)


Close-up: Screen_16_22 | upper_right_FOV1 | (10000, 2000, 12000, 4000)
Reusing: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/19_harmonized_annotations_over_HE/closeups/Screen_16_22/upper_right_FOV1/Screen_16_22_upper_right_FOV1_Specific_annotation_over_HE.png
Reusing: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/19_harmonized_annotations_over_HE/closeups/Screen_16_22/upper_right_FOV1/Screen_16_22_upper_right_FOV1_Broader_annotation_over_HE.png
Reusing: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/19_harmonized_annotations_over_HE/closeups/Screen_16_22/upper_right_FOV1/Screen_16_22_upper_right_FOV1_Specific_vs_Broader_over_HE.png
Close-up: Screen_16_22 | upper_middle_FOV1 | (6000, 4000, 8000, 6000)
Reusing: /host_root/nethome/reny28/Projects/Visium_projects/TBIO-8057/tmp/proseg_resolvi_immune_enrichment_v1/19_harmonized_a

,sample,roi_name,left,upper,right,lower,n_cells,specific,broader,comparison
0,Screen_16_22,upper_right_FOV1,10000,2000,12000,4000,2519,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
1,Screen_16_22,upper_middle_FOV1,6000,4000,8000,6000,2329,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
2,Screen_16_22,middle_left_FOV1,4000,6000,6000,8000,4486,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
3,Screen_16_22,lower_left_FOV1,4200,11000,6200,13000,2433,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
4,C2D15_16_22,upper_left_FOV1,3500,1000,5500,3000,4418,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
5,C2D15_16_22,upper_middle_FOV1,4000,2000,6000,4000,4057,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...
6,C2D15_16_22,lower_middle_FOV1,7000,7500,9000,9500,4373,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...,/host_root/nethome/reny28/Projects/Visium_proj...


In [14]:
# ---------------------------------------------------------------------
# Save reproducibility metadata and final inventory
# ---------------------------------------------------------------------
resolved_roi_payload = {
    sample: {
        name: list(bounds)
        for name, bounds in resolved_rois_for_sample(
            sample
        ).items()
    }
    for sample in SAMPLE_ORDER
    if resolved_rois_for_sample(
        sample
    )
}

run_metadata = {
    "pipeline_version": PIPELINE_VERSION,
    "merged_handoff": str(
        MERGED_HANDOFF_PATH
    ),
    "annotation_handoff": str(
        ANNOTATION_HANDOFF_PATH
    ),
    "palette_json": str(
        PALETTE_JSON_PATH
    ),
    "specific_column": SPECIFIC_COLUMN,
    "broader_column": BROADER_COLUMN,
    "cell_alpha": float(
        CELL_ALPHA
    ),
    "whole_point_size": float(
        WHOLE_POINT_SIZE
    ),
    "roi_point_size": float(
        ROI_POINT_SIZE
    ),
    "HE_brightness": float(
        HE_BRIGHTNESS
    ),
    "HE_contrast": float(
        HE_CONTRAST
    ),
    "HE_color_saturation": float(
        HE_COLOR_SATURATION
    ),
    "whole_preview_max_side": int(
        WHOLE_PREVIEW_MAX_SIDE
    ),
    "plot_dpi": int(
        PLOT_DPI
    ),
    "resolved_rois": (
        resolved_roi_payload
    ),
}

(
    OUTPUT_ROOT
    / "HE_overlay_run_metadata.json"
).write_text(
    json.dumps(
        run_metadata,
        indent=2,
    )
)

print("Output inventory:")
for path in sorted(
    OUTPUT_ROOT.rglob("*")
):
    if path.is_file():
        print(
            f"{path.relative_to(OUTPUT_ROOT)!s:110s} "
            f"{path.stat().st_size / 1024**2:9.2f} MiB"
        )

gc.collect()


Output inventory:
HE_overlay_preflight.csv                                                                                            0.00 MiB
HE_overlay_run_metadata.json                                                                                        0.00 MiB
ROI_image_dimensions_and_overviews.csv                                                                              0.00 MiB
ROI_selection_overviews/C2D15_16_22_ROI_selection_grid.png                                                         11.92 MiB
ROI_selection_overviews/C2D15_17_26_ROI_selection_grid.png                                                          9.22 MiB
ROI_selection_overviews/C2D15_18_23_ROI_selection_grid.png                                                          7.75 MiB
ROI_selection_overviews/C2D15_23_25_ROI_selection_grid.png                                                          5.75 MiB
ROI_selection_overviews/C2D15_30_16_ROI_selection_grid.png                                                 

89611

# Practical adjustment guide

## H&E is obscured

Reduce:

```python
CELL_ALPHA = 0.35
WHOLE_POINT_SIZE = 3.0
ROI_POINT_SIZE = 5.0
```

## Cells are too difficult to see

Increase:

```python
CELL_ALPHA = 0.65
WHOLE_POINT_SIZE = 7.0
ROI_POINT_SIZE = 10.0
```

## H&E is too dark behind the points

Try:

```python
HE_BRIGHTNESS = 1.15
HE_CONTRAST = 0.90
HE_COLOR_SATURATION = 0.85
```

These settings alter only the displayed background; they do not modify the
stored image.

## Faster whole-slide figures

Reduce:

```python
WHOLE_PREVIEW_MAX_SIDE = 4000
PLOT_DPI = 250
```

The native coordinate alignment remains unchanged.

## Add a close-up at 2/3 width and 1/4 height

```python
ROI_SPECS["Screen_30_16"] = {
    "upper_right_target": {
        "center_fraction": (2 / 3, 1 / 4),
        "size": 2500,
    }
}
```

Then rerun:

```text
ROI-selection-overview cell
close-up-generation cell
metadata/inventory cell
```

The whole-slide outputs do not need to be regenerated.

## Make only one annotation type

```python
MAKE_SEPARATE_SPECIFIC = True
MAKE_SEPARATE_BROADER = False
MAKE_SIDE_BY_SIDE = False
```

## Existing output reuse

With:

```python
OVERWRITE = False
```

existing non-empty PNGs are reused. Set:

```python
OVERWRITE = True
```

when changing point size, transparency, background appearance, or ROI bounds
and you want the corresponding figures replaced.
